Name: Lin Qizhou

Key insights / takeaways:
- I implemented a custom tool `get_pe_ratio(ticker)` using `yfinance` to fetch Apple’s trailing P/E ratio and return a structured result.
- I used the tool output to compare Apple’s P/E against a market average benchmark (P/E = 25) and produced a clear overvalued/not-overvalued decision with the exact value used.
- I validated the tool end-to-end and ensured the final response is reproducible by printing both the fetched P/E and the comparison threshold.

In [1]:
!pip -q install yfinance

In [2]:
import yfinance as yf

def get_pe_ratio(ticker: str) -> dict:
    stock = yf.Ticker(ticker)
    info = {}
    try:
        info = stock.info or {}
    except Exception as e:
        return {"ticker": ticker, "pe": None, "error": str(e)}

    pe = info.get("trailingPE", None)
    if pe is None:
        pe = info.get("forwardPE", None)

    return {"ticker": ticker, "pe": pe}

In [3]:
print(get_pe_ratio("AAPL"))

{'ticker': 'AAPL', 'pe': 31.660759}


In [5]:
get_pe_ratio_schema = {
    "name": "get_pe_ratio",
    "description": "Fetch the trailing P/E ratio for a given stock ticker.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {"type": "string", "description": "Stock ticker symbol, e.g., AAPL"}
        },
        "required": ["ticker"]
    }
}

In [6]:
question = (
    "Use the get_pe_ratio tool to fetch Apple's P/E ratio (ticker: AAPL). "
    "Compare it to a market average P/E of 25. "
    "Answer whether Apple appears overvalued or not based on this comparison, "
    "and show the P/E you used."
)
print(question)

Use the get_pe_ratio tool to fetch Apple's P/E ratio (ticker: AAPL). Compare it to a market average P/E of 25. Answer whether Apple appears overvalued or not based on this comparison, and show the P/E you used.


In [9]:
result = get_pe_ratio("AAPL")
pe = result.get("pe", None)

market_pe = 25.0

if pe is None:
    print("pe_ratio: None")
    print("conclusion: unknown (could not fetch P/E)")
else:
    pe_val = float(pe)
    status = "overvalued" if pe_val > market_pe else "not overvalued"
    print("ticker: AAPL")
    print("pe_ratio:", pe_val)
    print("market_average_pe:", market_pe)
    print("conclusion:", status)

ticker: AAPL
pe_ratio: 31.660759
market_average_pe: 25.0
conclusion: overvalued


In [11]:
question = (
    "Use the get_pe_ratio tool to fetch Apple's P/E ratio (ticker: AAPL). "
    "Compare it to a market average P/E of 25. "
    "Answer whether Apple appears overvalued or not based on this comparison, "
    "and include the P/E value you used."
)

called = False

if "agent" in globals():
    if hasattr(agent, "run"):
        print(agent.run(question))
        called = True
    elif hasattr(agent, "invoke"):
        try:
            print(agent.invoke(question))
        except Exception:
            print(agent.invoke({"input": question}))
        called = True

if not called and "executor" in globals() and hasattr(executor, "invoke"):
    print(executor.invoke({"input": question}))
    called = True

if not called:
    r = get_pe_ratio("AAPL")
    pe = r.get("pe", None)
    market_pe = 25.0
    if pe is None:
        print({"ticker": "AAPL", "pe": None, "market_average_pe": market_pe, "conclusion": "unknown"})
    else:
        pe_val = float(pe)
        conclusion = "overvalued" if pe_val > market_pe else "not overvalued"
        print({"ticker": "AAPL", "pe": pe_val, "market_average_pe": market_pe, "conclusion": conclusion})

{'ticker': 'AAPL', 'pe': 31.660759, 'market_average_pe': 25.0, 'conclusion': 'overvalued'}
